<a href="https://colab.research.google.com/github/matinapap/Multimedia-Systems/blob/main/Multimedia_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!python --version


Python 3.12.11


In [ ]:
import shutil
import os

# Folders to delete before a fresh run
folders_to_delete = ["results"]

for folder in folders_to_delete:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Deleted folder: {folder}")
    else:
        print(f"Folder not found, skipping: {folder}")

Deleted folder: results


### Input video:
On Colab, the video is read from Google Drive (`MyDrive/input_video.avi`). When running locally, set `video_path` to a video file on disk.

In [ ]:
# -------------------------------
# Input video location
# -------------------------------
try:
    # On Colab: mount Google Drive and read the video from it
    from google.colab import drive
    drive.mount('/content/drive')
    video_path = '/content/drive/MyDrive/input_video.avi'
except ImportError:
    # Locally: path to the video file
    video_path = 'input_video.avi'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# -------------------------------
# Imports
# -------------------------------
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import heapq
from collections import Counter

In [ ]:
# -------------------------------
# (A) Frame differences
# -------------------------------
"""
1. Opens the video
2. Reads the first frame and converts it to grayscale
3. Computes the differences between consecutive frames
4. Stores the difference images in a list
"""
def compute_frame_differences(video_path):
    # load the video
    cap = cv2.VideoCapture(video_path)

    # check that the video opens
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    ret, prev_frame = cap.read()
    if not ret:
        raise IOError("Cannot read the first frame from the video")

    # convert the first frame to grayscale
    prev_frame_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    # list that stores all frame differences
    frame_diffs = []

    while True:
        # read the next frame
        ret, frame = cap.read()
        if not ret:
            break
        # convert the current frame to grayscale
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # compute the absolute difference between the current and the previous frame
        diff = cv2.absdiff(frame_gray, prev_frame_gray)

        # store the difference image
        frame_diffs.append(diff)

        # update the previous frame
        prev_frame_gray = frame_gray

    # release the video capture object
    cap.release()

    return frame_diffs

def visualize_frame_differences(frame_diffs, output_dir, prefix="diff"):
    # make sure the output folder exists
    os.makedirs(output_dir, exist_ok=True)

    # loop over all difference frames
    for i, diff in enumerate(frame_diffs):
        plt.imshow(diff, cmap='gray') # show the difference image in grayscale
        plt.axis('off') # hide the X and Y axes
        plt.title(f'{prefix} {i}')  # label which difference image this is
        plt.savefig(os.path.join(output_dir, f'{prefix}_{i:04d}.png')) # save the image as PNG in the output directory
        plt.close()

In [ ]:
# -------------------------------
# (B) Huffman coding
# -------------------------------
"""
1. Computes the symbol frequencies
2. Builds the Huffman tree
3. Assigns codes and returns the statistics
"""
def compute_huffman(frame_diffs, output_dir="huffman_output"):
    # make sure the output folder exists
    os.makedirs(output_dir, exist_ok=True)

    # count symbol frequencies
    freq = Counter()
    total_symbols = 0
    for diff in frame_diffs:
        arr = diff.ravel()  # flatten the image to a 1D array
        freq.update(arr.tolist()) # count how many times each pixel value appears
        total_symbols += arr.size # count the total number of symbols

    # check that there are symbols
    if total_symbols == 0:
        print("No symbols found in frame_diffs.")
        return {}, 0, 0

    # build a min-heap priority queue of the symbols, keyed by frequency
    heap = []
    uid = 0
    # priority queue
    for sym, f in freq.items():
        heap.append((f, uid, sym))
        uid += 1
    heapq.heapify(heap)

    # build the Huffman tree
    if len(heap) == 1:
        _, _, only_sym = heap[0]
        codes = {int(only_sym): "0"}  # a single symbol gets the code 0
    else:
        while len(heap) > 1: # otherwise build the Huffman tree
            f1, id1, node1 = heapq.heappop(heap)
            f2, id2, node2 = heapq.heappop(heap)
            combined = (f1 + f2, uid, (node1, node2))
            uid += 1
            heapq.heappush(heap, combined)
        _, _, root = heap[0]


        # assign Huffman codes
        # traverse the tree, adding 0 when going left and 1 when going right
        codes = {}
        def assign_codes(node, prefix):
            if isinstance(node, int) or isinstance(node, np.integer):
                codes[int(node)] = prefix if prefix != "" else "0"
                return
            left, right = node
            assign_codes(left, prefix + "0")
            assign_codes(right, prefix + "1")
        assign_codes(root, "")

    compressed_bits = sum(freq[sym] * len(codes[int(sym)]) for sym in freq)  # bits needed with Huffman coding
    original_bits = total_symbols * 8 # bits needed if every pixel were stored as 8-bit grayscale

    # save to .txt, for each pixel value: the symbol (e.g. 0, 128, 255), its frequency, its Huffman code and the code length
    codebook_path = os.path.join(output_dir, "huffman_codebook.txt")
    with open(codebook_path, "w") as fh:
        fh.write("symbol\tfrequency\tcode\tcode_length\n")
        for sym, f in freq.most_common():
            fh.write(f"{int(sym)}\t{f}\t{codes[int(sym)]}\t{len(codes[int(sym)])}\n")

    # print the statistics: original size (bits), compressed size, compression ratio and average bits per symbol
    print(f"Huffman codebook saved to: {codebook_path}")
    print(f"Original bits: {original_bits}")
    print(f"Compressed bitstream size: {compressed_bits}")
    if compressed_bits > 0:
        print(f"Compression ratio: {original_bits / compressed_bits:.4f}")
        print(f"Average bits per symbol: {compressed_bits / total_symbols:.4f}")

    return codes, original_bits, compressed_bits

In [ ]:
# -------------------------------
# (C) Motion Compensation
# -------------------------------
"""
1. Reads the video frames
2. Performs block matching (finds how each block moved)
3. Computes and stores the residual differences
"""
def block_matching(prev_frame, curr_frame, block_size=16, search_range=8):
    # frame height and width
    h, w = curr_frame.shape
    # prediction frame, initially all zeros
    pred_frame = np.zeros_like(curr_frame)

    for y in range(0, h, block_size):
        for x in range(0, w, block_size):
            # skip blocks that extend past the frame edges
            if y + block_size > h or x + block_size > w:
                continue

            block = curr_frame[y:y+block_size, x:x+block_size] # current block
            best_sad = float("inf")
            best_match = (y, x)

            # search prev_frame around the same position, within search_range, for the best matching block
            for dy in range(-search_range, search_range+1):
                for dx in range(-search_range, search_range+1):
                    ref_y = y + dy
                    ref_x = x + dx
                    if (ref_y < 0 or ref_x < 0 or
                        ref_y + block_size > h or
                        ref_x + block_size > w):
                        continue
                    ref_block = prev_frame[ref_y:ref_y+block_size,
                                            ref_x:ref_x+block_size]
                    # SAD (sum of absolute differences) between the blocks; the block with the lowest SAD is the best match
                    sad = np.sum(np.abs(block.astype(int) - ref_block.astype(int)))
                    if sad < best_sad:
                        best_sad = sad
                        best_match = (ref_y, ref_x)

            ref_y, ref_x = best_match
            # place the best matching block in pred_frame
            pred_frame[y:y+block_size, x:x+block_size] = prev_frame[ref_y:ref_y+block_size,
                                                                    ref_x:ref_x+block_size]
    return pred_frame

# Reads the video frame by frame, runs block matching on each pair of frames and computes the difference between the current and the predicted frame
def compute_motion_compensation(video_path, block_size=32, search_range=8):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    ret, prev_frame = cap.read()
    if not ret:
        raise IOError("Cannot read the first frame from the video")

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    mc_diffs = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        pred = block_matching(prev_gray, curr_gray,
                              block_size=block_size,
                              search_range=search_range)
        diff = cv2.absdiff(curr_gray, pred)
        mc_diffs.append(diff)

        prev_gray = curr_gray

    cap.release()
    return mc_diffs

In [ ]:
output_dir = "results"

# (A) Frame differences
print("---------------------------- (A) Frame differences ----------------------------")
frame_diffs = compute_frame_differences(video_path)
print(f"Computed {len(frame_diffs)} frame differences.")

diffs_dir = os.path.join(output_dir, "frame_diffs_A")
visualize_frame_differences(frame_diffs, diffs_dir, prefix="diff")

---------------------------- (A) Frame differences ----------------------------
Computed 113 frame differences.


In [ ]:
# (B) Huffman coding
print("---------------------------- (B) Huffman coding ----------------------------")
huffman_dir = os.path.join(output_dir, "huffman_B")
compute_huffman(frame_diffs, huffman_dir)

---------------------------- (B) Huffman coding ----------------------------
Huffman codebook saved to: results/huffman_B/huffman_codebook.txt
Original bits: 1874534400
Compressed bitstream size: 690429588
Compression ratio: 2.7150
Average bits per symbol: 2.9466


({3: '0000',
  9: '000100',
  15: '0001010',
  45: '0001011000',
  46: '0001011001',
  32: '000101101',
  22: '00010111',
  5: '00011',
  31: '001000000',
  64: '00100000100',
  145: '0010000010100000',
  156: '00100000101000010',
  157: '00100000101000011',
  137: '001000001010001',
  123: '00100000101001',
  125: '00100000101010',
  126: '00100000101011',
  78: '001000001011',
  60: '00100000110',
  59: '00100000111',
  102: '0010000100000',
  108: '0010000100001',
  77: '001000010001',
  61: '00100001001',
  44: '0010000101',
  62: '00100001100',
  103: '0010000110100',
  107: '0010000110101',
  76: '001000011011',
  63: '00100001110',
  73: '001000011110',
  165: '001000011111000000',
  174: '0010000111110000010',
  175: '0010000111110000011',
  167: '001000011111000010',
  173: '0010000111110000110',
  176: '0010000111110000111',
  144: '0010000111110001',
  155: '00100001111100100',
  152: '00100001111100101',
  153: '00100001111100110',
  154: '00100001111100111',
  127: '001000

In [ ]:
# (C) Motion-compensated differences
print("---------------------------- (C) Motion-compensated differences ----------------------------")
mc_dir = os.path.join(output_dir, "mc_diffs_C")

# Compute the motion-compensated differences
mc_diffs = compute_motion_compensation(video_path,
                                       block_size=32,
                                       search_range=8)
print(f"Computed {len(mc_diffs)} MC frame differences.")

# Visualization
visualize_frame_differences(mc_diffs, mc_dir, prefix="mc_diff")

# Huffman coding of the MC differences
compute_huffman(mc_diffs, os.path.join(output_dir, "huffman_C"))

---------------------------- (C) Motion-compensated differences ----------------------------
Computed 113 MC frame differences.
Huffman codebook saved to: results/huffman_C/huffman_codebook.txt
Original bits: 1874534400
Compressed bitstream size: 497511204
Compression ratio: 3.7678
Average bits per symbol: 2.1232


({1: '00',
  7: '010000',
  15: '01000100',
  22: '010001010',
  31: '0100010110',
  32: '0100010111',
  9: '0100011',
  47: '01001000000',
  46: '01001000001',
  64: '010010000100',
  82: '0100100001010',
  85: '0100100001011',
  60: '010010000110',
  62: '010010000111',
  79: '0100100010000',
  121: '010010001000100',
  180: '0100100010001010000000',
  182: '0100100010001010000001',
  176: '010010001000101000001',
  166: '01001000100010100001',
  167: '01001000100010100010',
  170: '01001000100010100011',
  149: '010010001000101001',
  140: '01001000100010101',
  134: '0100100010001011',
  107: '01001000100011',
  63: '010010001001',
  61: '010010001010',
  59: '010010001011',
  30: '0100100011',
  21: '010010010',
  20: '010010011',
  14: '01001010',
  44: '01001011000',
  103: '01001011001000',
  102: '01001011001001',
  130: '0100101100101000',
  133: '0100101100101001',
  117: '010010110010101',
  138: '01001011001011000',
  159: '0100101100101100100',
  165: '0100101100101100101